In [ ]:
from wind_paths import DATA_DIR, MODEL_DIR, RESULT_DIR
from wind_io import open_data_nc
from wind_export_2 import create_nc
from pinn_paired_model import Auto_DA_TransUnet_pytorch

In [ ]:
from pathlib import Path
import os
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
import numpy as np
import pandas as pd
import torch
from IPython.display import display


RUN_TAG = 'pinn_followup_v1'
RESULT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

if not torch.cuda.is_available():
    raise RuntimeError('本 notebook 要求带 CUDA 的 PyTorch 内核；当前未发现 GPU。')
cuda_device = torch.device(os.environ.get('YANGTZE_DEVICE', 'cuda:0'))
if cuda_device.type != 'cuda':
    raise ValueError('This experiment requires a CUDA device.')
torch.cuda.set_device(cuda_device.index if cuda_device.index is not None else 0)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
print('CUDA device:', torch.cuda.current_device(), torch.cuda.get_device_name())

TIMESTEP, DOWNNUM, COVNUM = 6, 4, 2
BASE_LAMBDAS = {
    'continuity': 611.7192928848946,
    'vorticity': 142.40845542406902,
    'shear': 0.0,
}
TRAINING_CONFIG = {'test_size': 0.2, 'valid_size': 0.1, 'k_fold': None, 'task_mode': 'regression', 'if_best_mode': 'no', 'modelpath': None, 'ifrandom_split': 'no', 'if_bottom_attention_only': 'yes', 'if_resnet': 'yes', 'layer_time': 1, 'baselayer': 16, 'cov_kernelsize': 3, 'cov_strides': 1, 'pool_method': 'maxpooling', 'pool_kernel_size': 2, 'pool_strides': 2, 'ifnormalization': 'no', 'normalization_method': None, 'trans_num': 1, 'num_heads': 1, 'dim_feedforward': 128, 'trans_dropout_rate': 0.0, 'activate': 'tanh', 'nlp_activate': 'relu', 'if_last_act': 'no', 'if_print_model': 'yes', 'loss_function': 'default', 'optimizer': 'Adam', 'metrics': 'Pearsonr', 'learning_rate': 0.0001, 'epochs': 5000, 'batch_size': 500, 'if_early_stopping': 100, 'ifheatmap': 'no', 'ifweight': 'no', 'ifmute': 'no', 'ifsave': 'yes'}
FUNCTION_SOURCE_SHA256 = '0f4063c644a7eeb81e89b2039efd7811effa028e573de8f731fbab344fd7de24'
PREPROCESS_SOURCE_SHA256 = 'f7ec1a25bf3c8026afc1f86d4e2ef7cc9bf4cf68e8818c5754c4176c2e75d616'

In [ ]:
import xarray as xr

import numpy as np
sp,lon,lat,levels,latlow,lattop,lonleft,lonright,times=open_data_nc('one',str(DATA_DIR / 'Surface-pressure.nc'),'sp','yes','time','2000-01-01-06','2023-12-31-23','yes','longitude','yes','latitude',24.0,36.0,108.0,126.0,0.25,0.25,'no','no',None,None,changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
st,lon,lat,levels,latlow,lattop,lonleft,lonright,times=open_data_nc('one',str(DATA_DIR / '2m-temperature.nc'),'t2m','yes','time','2000-01-01-06','2023-12-31-23','yes','longitude','yes','latitude',24.0,36.0,108.0,126.0,0.25,0.25,'no','no',None,None,changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
sd,lon,lat,levels,latlow,lattop,lonleft,lonright,times=open_data_nc('one',str(DATA_DIR / '2m-dewpoint-temperature.nc'),'d2m','yes','time','2000-01-01-06','2023-12-31-23','yes','longitude','yes','latitude',24.0,36.0,108.0,126.0,0.25,0.25,'no','no',None,None,changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
u850,lon,lat,levels,latlow,lattop,lonleft,lonright,times=open_data_nc('one',str(DATA_DIR / 'U-component-of-wind-850hpa.nc'),'u','yes','time','2000-01-01-06','2023-12-31-23','yes','longitude','yes','latitude',24.0,36.0,108.0,126.0,0.25,0.25,'no','no',None,None,changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
u925,lon,lat,levels,latlow,lattop,lonleft,lonright,times=open_data_nc('one',str(DATA_DIR / 'U-component-of-wind-925hpa.nc'),'u','yes','time','2000-01-01-06','2023-12-31-23','yes','longitude','yes','latitude',24.0,36.0,108.0,126.0,0.25,0.25,'no','no',None,None,changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
u1000,lon,lat,levels,latlow,lattop,lonleft,lonright,times=open_data_nc('one',str(DATA_DIR / 'U-component-of-wind-1000hpa.nc'),'u','yes','time','2000-01-01-06','2023-12-31-23','yes','longitude','yes','latitude',24.0,36.0,108.0,126.0,0.25,0.25,'no','no',None,None,changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
v850,lon,lat,levels,latlow,lattop,lonleft,lonright,times=open_data_nc('one',str(DATA_DIR / 'V-component-of-wind-850hpa.nc'),'v','yes','time','2000-01-01-06','2023-12-31-23','yes','longitude','yes','latitude',24.0,36.0,108.0,126.0,0.25,0.25,'no','no',None,None,changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
v925,lon,lat,levels,latlow,lattop,lonleft,lonright,times=open_data_nc('one',str(DATA_DIR / 'V-component-of-wind-925hpa.nc'),'v','yes','time','2000-01-01-06','2023-12-31-23','yes','longitude','yes','latitude',24.0,36.0,108.0,126.0,0.25,0.25,'no','no',None,None,changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
v1000,lon,lat,levels,latlow,lattop,lonleft,lonright,times=open_data_nc('one',str(DATA_DIR / 'V-component-of-wind-1000hpa.nc'),'v','yes','time','2000-01-01-06','2023-12-31-23','yes','longitude','yes','latitude',24.0,36.0,108.0,126.0,0.25,0.25,'no','no',None,None,changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
w850,lon,lat,levels,latlow,lattop,lonleft,lonright,times=open_data_nc('one',str(DATA_DIR / 'Vertical-velocity-850hpa.nc'),'w','yes','time','2000-01-01-06','2023-12-31-23','yes','longitude','yes','latitude',24.0,36.0,108.0,126.0,0.25,0.25,'no','no',None,None,changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
w925,lon,lat,levels,latlow,lattop,lonleft,lonright,times=open_data_nc('one',str(DATA_DIR / 'Vertical-velocity-925hpa.nc'),'w','yes','time','2000-01-01-06','2023-12-31-23','yes','longitude','yes','latitude',24.0,36.0,108.0,126.0,0.25,0.25,'no','no',None,None,changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
w1000,lon,lat,levels,latlow,lattop,lonleft,lonright,times=open_data_nc('one',str(DATA_DIR / 'Vertical-velocity-1000hpa.nc'),'w','yes','time','2000-01-01-06','2023-12-31-23','yes','longitude','yes','latitude',24.0,36.0,108.0,126.0,0.25,0.25,'no','no',None,None,changeresolution=1,timespace=1,ifchange_west_east='no',ifinterpolate='no')
w850=w850*-0.1
w925=w925*-0.1
w1000=w1000*-0.1

In [ ]:
import numpy as np
vx=np.zeros((3,u850.shape[0]-1,u850.shape[1],u850.shape[2]),dtype='float32')
vy=np.zeros((9,u850.shape[0]-1,u850.shape[1],u850.shape[2]),dtype='float32')
vx[0,:,:]=sp[:-1,:,:]
vx[1,:,:]=st[:-1,:,:]
vx[2,:,:]=sd[:-1,:,:]
vy[0,:,:,:]=u850[1:,:,:]
vy[1,:,:,:]=u925[1:,:,:]
vy[2,:,:,:]=u1000[1:,:,:]
vy[3,:,:,:]=v850[1:,:,:]
vy[4,:,:,:]=v925[1:,:,:]
vy[5,:,:,:]=v1000[1:,:,:]
vy[6,:,:,:]=w850[1:,:,:]
vy[7,:,:,:]=w925[1:,:,:]
vy[8,:,:,:]=w1000[1:,:,:]
vx=vx.transpose(1,2,3,0)
vy=vy.transpose(1,2,3,0)
print(vx.shape,vy.shape)
window_offset = 5
test_size, valid_size = 0.2, 0.1
test_start = int((1 - test_size) * len(vy))
train_stop = int((1 - valid_size / (1 - test_size)) * test_start)
train_inputs = vx[:train_stop + window_offset]
train_labels = vy[window_offset:train_stop + window_offset]
vx_mean = np.nanmean(train_inputs, axis=0)
vx_std = np.nanstd(train_inputs, axis=0)
vy_mean = np.nanmean(train_labels, axis=0)
vy_std = np.nanstd(train_labels, axis=0)
vx = (vx - vx_mean) / vx_std
vy = (vy - vy_mean) / vy_std
del train_inputs, train_labels

In [ ]:
from pathlib import Path
import gc
import hashlib
import json
import os
import random
import time
import uuid
import numpy as np
import pandas as pd
import torch


def _json_value(value):
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    raise TypeError(type(value).__name__)


def _write_json_atomic(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_name(path.name + '.' + uuid.uuid4().hex + '.tmp')
    try:
        temp.write_text(
            json.dumps(value, ensure_ascii=False, indent=2, default=_json_value),
            encoding='utf-8'
        )
        os.replace(temp, path)
    finally:
        if temp.exists():
            temp.unlink()


def _configuration_hash(value):
    raw = json.dumps(
        value, sort_keys=True, ensure_ascii=False,
        separators=(',', ':'), default=_json_value
    ).encode('utf-8')
    return hashlib.sha256(raw).hexdigest()


def _array_sha256(value, chunk=128):
    value = np.asarray(value)
    digest = hashlib.sha256()
    digest.update(str(value.dtype).encode())
    digest.update(str(value.shape).encode())
    if value.ndim == 0:
        digest.update(value.tobytes())
    else:
        for start in range(0, value.shape[0], chunk):
            block = np.ascontiguousarray(value[start:start + chunk])
            digest.update(memoryview(block).cast('B'))
    return digest.hexdigest()


def prepare_followup_data(vx, vy, times, lat, lon, vy_mean, vy_std):
    levels = np.asarray([850, 925, 1000], dtype=np.float32)
    times = np.asarray(times, dtype='datetime64[ns]')
    if len(times) != len(vy) + 1 or len(vx) != len(vy):
        raise ValueError('当前数据长度不符合原 notebook 的输入[:-1]、目标[1:]对应关系。')
    if vx.shape[-1] != 3 or vy.shape[-1] != 9:
        raise ValueError('输入/输出通道应为原来的 3/9。')
    target_times = times[1:][TIMESTEP - 1:]
    test_start = int((1 - TRAINING_CONFIG['test_size']) * len(vy))
    train_end = int(
        (1 - TRAINING_CONFIG['valid_size'] /
         (1 - TRAINING_CONFIG['test_size'])) * test_start
    )
    test_times = target_times[test_start:]
    if len(test_times) == 0:
        raise ValueError('测试集为空。')
    if np.isnat(times).any() or np.any(np.diff(times.astype('int64')) <= 0):
        raise ValueError('原始时间必须唯一、严格递增且不含 NaT。')

    print('记录输入和目标的完整数据指纹（分块计算，不生成完整副本）……')
    data_record = {
        'vx_shape': list(vx.shape),
        'vy_shape': list(vy.shape),
        'vx_sha256': _array_sha256(vx),
        'vy_sha256': _array_sha256(vy),
        'times_sha256': _array_sha256(times.astype('int64')),
        'latitude_sha256': _array_sha256(np.asarray(lat)),
        'longitude_sha256': _array_sha256(np.asarray(lon)),
        'vy_mean_sha256': _array_sha256(vy_mean),
        'vy_std_sha256': _array_sha256(vy_std),
        'preprocessing_source_sha256': PREPROCESS_SOURCE_SHA256,
        'standardization_scope': 'training_input_and_target_rows',
        'window_count': int(len(target_times)),
        'train_count': int(train_end),
        'validation_count': int(test_start - train_end),
        'test_count': int(len(test_times)),
        'test_first_time': str(test_times[0]),
        'test_last_time': str(test_times[-1]),
        'w_external_scale': -0.1,
        'levels_hpa': levels.tolist(),
    }
    fingerprint = _configuration_hash(data_record)
    normalization_path = RESULT_DIR / (
        f'da_transunet_1hour_{RUN_TAG}_normalization_{fingerprint[:12]}.npz'
    )
    if not normalization_path.exists():
        temp = normalization_path.with_name(
            normalization_path.name + '.' + uuid.uuid4().hex + '.tmp'
        )
        try:
            with temp.open('wb') as stream:
                np.savez_compressed(
                    stream, vx_mean=vx_mean, vx_std=vx_std,
                    vy_mean=vy_mean, vy_std=vy_std,
                    lat=np.asarray(lat), lon=np.asarray(lon), levels=levels,
                    test_times=test_times,
                    dataset_fingerprint=np.asarray(fingerprint),
                    w_external_scale=np.asarray(-0.1),
                )
            os.replace(temp, normalization_path)
        finally:
            if temp.exists():
                temp.unlink()
    _write_json_atomic(
        normalization_path.with_suffix('.json'),
        dict(data_record, dataset_fingerprint=fingerprint)
    )
    print('样本数 train / validation / test：',
          train_end, test_start - train_end, len(test_times))
    return dict(
        fingerprint=fingerprint, record=data_record,
        normalization_path=normalization_path,
        test_times=test_times, levels=levels
    )


def _release_cuda(model=None):
    if model is not None:
        models_to_clear = list(model) if isinstance(model, (list, tuple)) else [model]
        for current in models_to_clear:
            if current is None:
                continue
            for module in list(current.modules()):
                for key, value in list(module.__dict__.items()):
                    if key in ('_parameters', '_buffers', '_modules'):
                        continue
                    if torch.is_tensor(value) or isinstance(value, np.ndarray):
                        module.__dict__.pop(key, None)
                module.__dict__.pop('self', None)
                module.__dict__.pop('align_residual', None)
            current.to('cpu')
        models_to_clear.clear()
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


def _run_paths(run_id):
    result_stem = f'da_transunet_1hour_{RUN_TAG}_{run_id}'
    model_stem = MODEL_DIR / (
        f'DA_TransUNet_1_24year_changjiang_{RUN_TAG}_{run_id}'
    )
    history = RESULT_DIR / f'{result_stem}_loss_history.csv'
    paths = {
        'manifest': str(RESULT_DIR / f'{result_stem}_manifest.json'),
        'loss_history': str(history),
        'loss_summary': str(history.with_suffix('.summary.json')),
        'model': str(model_stem) + '.pth',
        'optimizer': str(model_stem) + '_opt.pth',
        'normalization': str(DATA_CONTEXT['normalization_path']),
    }
    for kind in ['testy', 'predicty']:
        for component in ['u', 'v', 'w']:
            name = f'{kind}_{component}'
            paths[name] = str(RESULT_DIR / (
                f'da_transunet_{name}_1_24year_changjiang_{RUN_TAG}_{run_id}.nc'
            ))
    return paths, model_stem


def _save_pair_nc(path, variable, values):
    values = np.asarray(values)
    if values.shape[0] != len(DATA_CONTEXT['test_times']):
        raise ValueError('保存数组与精确映射的测试时次长度不同。')
    limit = np.finfo(np.float32).max
    for start in range(0, len(values), 128):
        block = values[start:start + 128]
        if not np.isfinite(block).all() or np.any(np.abs(block) > limit):
            raise ValueError(f'{variable} 在 {start}:{start + 128} 包含非法值；停止保存。')
    path = Path(path)
    temp = path.with_name(path.stem + '.' + uuid.uuid4().hex + '.partial.nc')
    try:
        create_nc(
            DATA_CONTEXT['test_times'], lat, lon, DATA_CONTEXT['levels'],
            values, 'time', 'latitude', 'longitude', variable,
            'all', 'level', 'yes', 'yes', 'yes', str(temp)
        )
        import netCDF4
        with netCDF4.Dataset(temp, 'a') as ds:
            ds.dataset_fingerprint = DATA_CONTEXT['fingerprint']
            ds.variables['level'].units = 'hPa'
        os.replace(temp, path)
    finally:
        if temp.exists():
            temp.unlink()


def _assert_pair_matches(completed):
    groups = {}
    for record in completed:
        key = int(record['seed'])
        signature = (
            record['dataset_fingerprint'],
            record['init_state_sha256'],
            record['testy_standardized_sha256']
        )
        if key in groups and groups[key] != signature:
            raise RuntimeError(
                f'seed={key} 的数据、初始参数或 testy 不一致，不能作为配对比较。'
            )
        groups[key] = signature


def run_followup_experiment(run_id, experiment, seed, global_scale):
    paths, model_stem = _run_paths(run_id)
    run_config = {
        'seed': int(seed), 'experiment': experiment,
        'global_scale': float(global_scale),
        'lambda_continuity': BASE_LAMBDAS['continuity'],
        'lambda_vorticity': BASE_LAMBDAS['vorticity'],
        'lambda_shear': BASE_LAMBDAS['shear'],
        'timestep': TIMESTEP, 'downnum': DOWNNUM, 'covnum': COVNUM,
        'training': TRAINING_CONFIG,
        'function_source_sha256': FUNCTION_SOURCE_SHA256,
        'dataset_fingerprint': DATA_CONTEXT['fingerprint'],
        'torch_version': torch.__version__,
        'numpy_version': np.__version__,
        'cuda_version': torch.version.cuda,
        'cudnn_deterministic': bool(torch.backends.cudnn.deterministic),
        'cudnn_benchmark': bool(torch.backends.cudnn.benchmark),
        'deterministic_algorithms': bool(torch.are_deterministic_algorithms_enabled()),
    }
    config_fingerprint = _configuration_hash(run_config)
    manifest_path = Path(paths['manifest'])
    if manifest_path.exists():
        previous = json.loads(manifest_path.read_text(encoding='utf-8'))
        if previous.get('config_fingerprint') != config_fingerprint:
            raise RuntimeError(
                f'{run_id} 已有不同设置的结果。请使用新的 RUN_TAG，保留已有结果。'
            )
        if previous.get('status') == 'completed':
            required = [
                'model', 'optimizer', 'loss_history', 'loss_summary', 'normalization',
                'testy_u', 'testy_v', 'testy_w', 'predicty_u', 'predicty_v', 'predicty_w'
            ]
            if all(Path(paths[key]).is_file() and Path(paths[key]).stat().st_size
                   for key in required):
                print('复用已完整保存且设置一致的实验：', run_id)
                return previous
            raise RuntimeError(f'{run_id} 标记完成但有输出缺失，请核对对应 manifest。')
        print('上次实验未完整完成，本次从头重跑：', run_id)
    else:
        orphan = [str(value) for key, value in paths.items()
                  if key not in ('manifest', 'normalization') and Path(value).exists()]
        if orphan:
            raise FileExistsError('发现无 manifest 的同名输出，请更换 RUN_TAG：' + orphan[0])

    record = {
        'schema_version': 1, 'status': 'running', 'run_id': run_id,
        'experiment': experiment, 'seed': int(seed), 'global_scale': float(global_scale),
        'lambda_continuity': BASE_LAMBDAS['continuity'],
        'lambda_vorticity': BASE_LAMBDAS['vorticity'],
        'lambda_shear': BASE_LAMBDAS['shear'],
        'effective_weights': {
            key: float(global_scale) * value for key, value in BASE_LAMBDAS.items()
        },
        'dataset_fingerprint': DATA_CONTEXT['fingerprint'],
        'config_fingerprint': config_fingerprint,
        'config': run_config, 'paths': paths,
        'started_at': time.strftime('%Y-%m-%dT%H:%M:%S'),
        'checkpoint_note': 'Model weights are restored to selected epoch; optimizer is not a resume checkpoint.',
    }
    _write_json_atomic(manifest_path, record)
    model = outputs = prediction = truth = physical = None
    _release_cuda()
    print(f'\n开始 {run_id}：seed={seed}, overall multiplier={global_scale:g}')
    try:
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        outputs = Auto_DA_TransUnet_pytorch(
            vy, vx, TIMESTEP, DOWNNUM, COVNUM,
            **TRAINING_CONFIG,
            pinn_vy_mean=vy_mean, pinn_vy_std=vy_std,
            pinn_lat=lat, pinn_lon=lon,
            pinn_pressure_levels=DATA_CONTEXT['levels'],
            pinn_w_external_scale=-0.1,
            pinn_lambda_continuity=BASE_LAMBDAS['continuity'],
            pinn_lambda_vorticity=BASE_LAMBDAS['vorticity'],
            pinn_lambda_shear=BASE_LAMBDAS['shear'],
            pinn_global_scale=float(global_scale),
            pinn_random_seed=int(seed),
            pinn_history_path=paths['loss_history'],
            pinn_run_metadata={
                'run_id': run_id, 'experiment': experiment,
                'dataset_fingerprint': DATA_CONTEXT['fingerprint'],
                'config_fingerprint': config_fingerprint,
            },
            savepath=str(model_stem), device='gpu'
        )
        model, prediction, truth = outputs[:3]
        expected = (len(DATA_CONTEXT['test_times']), 9, len(lat), len(lon))
        if truth.shape != expected or prediction.shape != expected:
            raise ValueError(f'返回数组形状不符合原数据时次映射：{truth.shape}, {prediction.shape}。')
        record['testy_standardized_sha256'] = _array_sha256(truth)

        for kind, array in [('testy', truth), ('predicty', prediction)]:
            for component, channel_start in [('u', 0), ('v', 3), ('w', 6)]:
                channels = slice(channel_start, channel_start + 3)
                physical = array[:, channels].transpose(0, 2, 3, 1)
                physical = physical * vy_std[..., channels] + vy_mean[..., channels]
                variable = f'{kind}_{component}'
                _save_pair_nc(paths[variable], variable, physical.transpose(0, 3, 1, 2))
                physical = None
                gc.collect()

        summary = json.loads(Path(paths['loss_summary']).read_text(encoding='utf-8'))
        if not summary['checkpoint_restored']:
            raise RuntimeError('当前模型没有恢复已选定的最佳 epoch。')
        record['init_state_sha256'] = summary['init_state_sha256']
        record['best_epoch'] = int(summary['best_epoch'])
        record['selected_validation_data_loss'] = float(
            summary['selected_epoch_record']['validation_data_loss']
        )
        record['status'] = 'completed'
        record['completed_at'] = time.strftime('%Y-%m-%dT%H:%M:%S')
        record['file_sizes'] = {
            key: Path(path).stat().st_size
            for key, path in paths.items() if key != 'manifest'
        }
        _write_json_atomic(manifest_path, record)
        print('完成：', run_id, '；selected epoch =', record['best_epoch'])
        return record
    except Exception as exc:
        record['status'] = 'failed'
        record['error'] = f'{type(exc).__name__}: {exc}'
        _write_json_atomic(manifest_path, record)
        raise
    finally:
        _release_cuda(model)
        model = outputs = prediction = truth = physical = None
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            print('本次对象释放后 CUDA allocated / reserved (GiB)：',
                  round(torch.cuda.memory_allocated(0) / 2**30, 3),
                  round(torch.cuda.memory_reserved(0) / 2**30, 3))


def execute_followup_suite(suite_id, specifications, index_name):
    index_path = RESULT_DIR / index_name
    manifest_paths = [_run_paths(item['run_id'])[0]['manifest'] for item in specifications]
    _write_json_atomic(index_path, {
        'schema_version': 1, 'suite_id': suite_id,
        'runs': manifest_paths, 'run_tag': RUN_TAG,
        'sensitivity_selection': 'validation_data_loss_at_selected_checkpoint',
    })
    completed = []
    for item in specifications:
        completed.append(run_followup_experiment(**item))
        _assert_pair_matches(completed)
    rows = [{
        'run_id': item['run_id'], 'seed': item['seed'],
        'experiment': item['experiment'], 'global_scale': item['global_scale'],
        'best_epoch': item['best_epoch'],
        'validation_data_loss': item['selected_validation_data_loss'],
        'manifest': item['paths']['manifest'],
    } for item in completed]
    table = pd.DataFrame(rows)
    table.to_csv(index_path.with_suffix('.csv'), index=False, encoding='utf-8-sig')
    display(table)
    print('实验索引：', index_path)
    return completed

In [ ]:
DATA_CONTEXT = prepare_followup_data(vx, vy, times, lat, lon, vy_mean, vy_std)
print('标准化参数：', DATA_CONTEXT['normalization_path'])

In [ ]:
EXPERIMENTS = [{'run_id': 'seed42_no_pinn', 'experiment': 'no_pinn', 'seed': 42, 'global_scale': 0.0}, {'run_id': 'seed42_no_shear', 'experiment': 'no_shear', 'seed': 42, 'global_scale': 1.0}, {'run_id': 'seed2026_no_pinn', 'experiment': 'no_pinn', 'seed': 2026, 'global_scale': 0.0}, {'run_id': 'seed2026_no_shear', 'experiment': 'no_shear', 'seed': 2026, 'global_scale': 1.0}]
completed_runs = execute_followup_suite(
    'seed_repeats', EXPERIMENTS,
    'da_transunet_1hour_pinn_seed_repeats_v1_index.json'
)